# **Definición del problema y dataset INMET**

**Objetivo:** describir el problema de forecasting meteorológico horario y la fuente de datos.




## **1. Definición formal**

Sea $y_t \in \mathbb{R}^T$ la(s) variable(s) objetivo y
$\mathbf{x}_t \in \mathbb{R}^F$ las covariables observadas. Buscamos un
modelo $f_\theta$ tal que
$$
\hat{y}_{t+1:t+H} = f_\theta\bigl(\mathbf{x}_{t-L+1:t},\, y_{t-L+1:t}\bigr),
$$
que minimice $\mathrm{RMSE}$ (con MAE/R² complementarias) en el conjunto de
test temporal.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

from src.utils import load_yaml, set_seed
cfg = load_yaml(PROJECT_ROOT / "config" / "config.yaml")
set_seed(cfg['project']['seed'])
cfg['task']

{'target': 'temp_c',
 'multi_target': [],
 'exog': ['humidity_pct', 'pressure_mb', 'radiation_kj_m2', 'wind_speed_ms'],
 'freq': 'h',
 'lookback': 168,
 'horizon': 168}

## **2. Dataset INMET**



In [2]:
from src.data.ingest_inmet import ingest


In [3]:
from src.data.clean import clean_station

interim_dir = PROJECT_ROOT / cfg['paths']['data_interim']
station = next(
    (p for p in sorted(interim_dir.iterdir()) if p.is_dir() and any(p.glob("*.csv"))),
    None,
)
if station is not None:
    df = clean_station(station, cfg)
    df.head()
else:
    print(f"Sin carpetas de estacion con CSV en {interim_dir}")

[2026-05-22 08:51:59] INFO    src.data.clean :: Estación A001 — 61368 filas, 6.3% NaN tras limpieza
